# A8 calibration and locked evaluation
Run calibration before the locked test is copied into the runtime. All logic is in `sipature_ml.evaluation`; this notebook only orchestrates the two immutable phases.

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

assert torch.cuda.is_available(), "A8 membutuhkan runtime GPU"

PyTorch: 2.11.0+cu128
CUDA tersedia: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import base64
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import userdata

REPOSITORY_URL = "https://github.com/jodypangaribuan/hackathon.git"
REPOSITORY_DIR = Path("/content/hackathon")
EXPECTED_COMMIT = "9eaca4f2780cdb88d2381fa558dd9bd445297c9b"

token = userdata.get("GITHUB_TOKEN")
assert token, (
    "GITHUB_TOKEN tidak ditemukan. "
    "Periksa Colab Secrets dan aktifkan Notebook access."
)

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

git_environment = os.environ.copy()
git_environment["GIT_CONFIG_COUNT"] = "1"
git_environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_environment["GIT_CONFIG_VALUE_0"] = (
    f"Authorization: Basic {authorization}"
)

# Hanya membersihkan clone lokal yang gagal, bukan Google Drive.
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)

clone = subprocess.run(
    [
        "git",
        "clone",
        REPOSITORY_URL,
        str(REPOSITORY_DIR),
    ],
    env=git_environment,
    text=True,
    capture_output=True,
)

print("Clone return code:", clone.returncode)
print(clone.stdout)
print(clone.stderr)

assert clone.returncode == 0, (
    "Clone gagal. Periksa izin GITHUB_TOKEN terhadap repository."
)

checkout = subprocess.run(
    [
        "git",
        "-C",
        str(REPOSITORY_DIR),
        "checkout",
        EXPECTED_COMMIT,
    ],
    env=git_environment,
    text=True,
    capture_output=True,
)

print("Checkout return code:", checkout.returncode)
print(checkout.stdout)
print(checkout.stderr)

assert checkout.returncode == 0, "Checkout commit A8 gagal"

actual_commit = subprocess.run(
    [
        "git",
        "-C",
        str(REPOSITORY_DIR),
        "rev-parse",
        "HEAD",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

worktree_status = subprocess.run(
    [
        "git",
        "-C",
        str(REPOSITORY_DIR),
        "status",
        "--porcelain",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Commit aktual:", actual_commit)
print("Commit sesuai:", actual_commit == EXPECTED_COMMIT)
print("Working tree bersih:", worktree_status == "")

assert actual_commit == EXPECTED_COMMIT
assert worktree_status == ""

Clone return code: 0

Cloning into '/content/hackathon'...

Checkout return code: 0

Note: switching to '9eaca4f2780cdb88d2381fa558dd9bd445297c9b'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 9eaca4f refactor: formalize immutable two-phase calibration and evaluation runbook with state tracking and hardware requirements

Commit aktual: 9eaca4f2780cdb88d2381fa558dd9bd445297c9b
Commit sesuai: True
Working tree bersih: True


In [7]:
%cd /content/hackathon/ml

import subprocess
import sys

commands = [
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchvision",
    ],
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements-colab.lock.txt",
    ],
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "-e",
        ".",
    ],
]

for command in commands:
    print("\nMenjalankan:", " ".join(command))
    result = subprocess.run(command)
    assert result.returncode == 0, (
        f"Perintah gagal dengan return code {result.returncode}: "
        f"{' '.join(command)}"
    )

print("\nSeluruh dependency berhasil dipasang.")
print("RESTART RUNTIME SEKARANG.")

/content/hackathon/ml

Menjalankan: /usr/bin/python3 -m pip uninstall -y torchvision

Menjalankan: /usr/bin/python3 -m pip install -r requirements-colab.lock.txt

Menjalankan: /usr/bin/python3 -m pip install --no-deps -e .

Seluruh dependency berhasil dipasang.
RESTART RUNTIME SEKARANG.


In [1]:
import importlib.util
import sys

import accelerate
import datasets
import numpy
import sklearn
import torch
import transformers
import yaml

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("NumPy:", numpy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyYAML:", yaml.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else None,
)
print(
    "Torchvision terpasang:",
    importlib.util.find_spec("torchvision") is not None,
)

assert torch.__version__.startswith("2.7.1")
assert transformers.__version__ == "4.53.2"
assert datasets.__version__ == "3.6.0"
assert accelerate.__version__ == "1.8.1"
assert numpy.__version__ == "2.2.6"
assert sklearn.__version__ == "1.7.2"
assert torch.cuda.is_available()
assert importlib.util.find_spec("torchvision") is None

print("\nEnvironment A8 sesuai dan GPU siap.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.7.1+cu126
Transformers: 4.53.2
Datasets: 3.6.0
Accelerate: 1.8.1
NumPy: 2.2.6
Scikit-learn: 1.7.2
PyYAML: 6.0.3
CUDA tersedia: True
GPU: Tesla T4
Torchvision terpasang: False

Environment A8 sesuai dan GPU siap.


In [4]:
import json
from pathlib import Path

MODEL_RUN_DIR = Path(
    "/content/drive/MyDrive/SIPATURE/runs/"
    "20260801-1024_indobert-silver-v1"
)

print("Folder model A7:", MODEL_RUN_DIR)
print("Folder ditemukan:", MODEL_RUN_DIR.is_dir())

assert MODEL_RUN_DIR.is_dir(), (
    "Folder model A7 tidak ditemukan. "
    "Periksa lokasi run di Google Drive."
)

required_paths = [
    "manifest.json",
    "summary.json",
    "aspect/model/config.json",
    "aspect/model/model.safetensors",
    "aspect/model/tokenizer_config.json",
    "aspect/model/special_tokens_map.json",
    "aspect/model/vocab.txt",
    "polarity/model/config.json",
    "polarity/model/model.safetensors",
    "polarity/model/tokenizer_config.json",
    "polarity/model/special_tokens_map.json",
    "polarity/model/vocab.txt",
]

missing_paths = []

for relative_path in required_paths:
    path = MODEL_RUN_DIR / relative_path
    exists = path.is_file()
    size = path.stat().st_size if exists else None

    print(
        f"{relative_path}:",
        "ADA" if exists else "HILANG",
        f"({size:,} byte)" if size is not None else "",
    )

    if not exists:
        missing_paths.append(relative_path)

assert not missing_paths, (
    "Artifact A7 tidak lengkap: " + ", ".join(missing_paths)
)

manifest = json.loads(
    (MODEL_RUN_DIR / "manifest.json").read_text(encoding="utf-8")
)
summary = json.loads(
    (MODEL_RUN_DIR / "summary.json").read_text(encoding="utf-8")
)

print("\nRun ID:", summary.get("run_id"))
print("Status:", summary.get("status"))
print("Test dibaca saat A7:", summary.get("test_read"))
print(
    "Jumlah hash artifact:",
    len(manifest.get("artifact_hashes", {})),
)
print(
    "Aspect offline reload:",
    summary.get("offline_reload_smoke", {}).get("aspect"),
)
print(
    "Polarity offline reload:",
    summary.get("offline_reload_smoke", {}).get("polarity"),
)
print(
    "Severity status:",
    summary.get("tasks", {}).get("severity", {}).get("status"),
)

assert summary.get("run_id") == (
    "20260801-1024_indobert-silver-v1"
)
assert summary.get("test_read") is False
assert len(manifest.get("artifact_hashes", {})) == 65
assert (
    summary["offline_reload_smoke"]["aspect"]["passed"]
    is True
)
assert (
    summary["offline_reload_smoke"]["polarity"]["passed"]
    is True
)
assert (
    summary["tasks"]["severity"]["status"]
    == "skipped_insufficient_support"
)

print("\nStruktur dan metadata model A7 sesuai.")

Folder model A7: /content/drive/MyDrive/SIPATURE/runs/20260801-1024_indobert-silver-v1
Folder ditemukan: True
manifest.json: ADA (8,533 byte)
summary.json: ADA (3,377 byte)
aspect/model/config.json: ADA (1,547 byte)
aspect/model/model.safetensors: ADA (497,831,984 byte)
aspect/model/tokenizer_config.json: ADA (1,292 byte)
aspect/model/special_tokens_map.json: ADA (125 byte)
aspect/model/vocab.txt: ADA (229,167 byte)
polarity/model/config.json: ADA (1,044 byte)
polarity/model/model.safetensors: ADA (497,798,148 byte)
polarity/model/tokenizer_config.json: ADA (1,292 byte)
polarity/model/special_tokens_map.json: ADA (125 byte)
polarity/model/vocab.txt: ADA (229,167 byte)

Run ID: 20260801-1024_indobert-silver-v1
Status: trained_train_validation_only
Test dibaca saat A7: False
Jumlah hash artifact: 65
Aspect offline reload: {'local_files_only': True, 'passed': True, 'shape': [1, 14]}
Polarity offline reload: {'local_files_only': True, 'passed': True, 'shape': [1, 3]}
Severity status: skipp

In [5]:
from pathlib import Path

split_candidates = [
    Path("/content/drive/MyDrive/SIPATURE/frozen-splits"),
    Path("/content/drive/MyDrive/SIPATURE/inputs/splits"),
]

available_split_dirs = []

for candidate in split_candidates:
    print("\nFolder:", candidate)
    print("Ditemukan:", candidate.is_dir())

    if candidate.is_dir():
        available_split_dirs.append(candidate)

        print("Nama file:")
        for path in sorted(candidate.iterdir()):
            if path.is_file():
                print("-", path.name)

assert available_split_dirs, (
    "Tidak ada folder frozen split yang ditemukan di Drive."
)


Folder: /content/drive/MyDrive/SIPATURE/frozen-splits
Ditemukan: False

Folder: /content/drive/MyDrive/SIPATURE/inputs/splits
Ditemukan: True
Nama file:
- split_manifest_silver_v1.json
- train_silver_v1.jsonl
- validation_silver_v1.jsonl


In [6]:
import hashlib
import json
from pathlib import Path

DRIVE_SPLIT_DIR = Path(
    "/content/drive/MyDrive/SIPATURE/inputs/splits"
)

MANIFEST_SOURCE = (
    DRIVE_SPLIT_DIR / "split_manifest_silver_v1.json"
)
VALIDATION_SOURCE = (
    DRIVE_SPLIT_DIR / "validation_silver_v1.jsonl"
)

assert MANIFEST_SOURCE.is_file()
assert VALIDATION_SOURCE.is_file()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

manifest = json.loads(
    MANIFEST_SOURCE.read_text(encoding="utf-8")
)

validation_entry = manifest.get("outputs", {}).get("validation")

assert isinstance(validation_entry, dict)
assert "path" in validation_entry
assert "sha256" in validation_entry

manifest_sha256 = sha256_file(MANIFEST_SOURCE)
validation_sha256 = sha256_file(VALIDATION_SOURCE)
expected_validation_sha256 = validation_entry["sha256"]

print("Split version:", manifest.get("split_version"))
print("Test dikunci:", manifest.get("test_is_locked"))
print("Validation path dalam manifest:", validation_entry["path"])
print("Manifest SHA-256:", manifest_sha256)
print("Validation SHA-256 aktual:", validation_sha256)
print(
    "Validation SHA-256 manifest:",
    expected_validation_sha256,
)
print(
    "Validation hash sesuai:",
    validation_sha256 == expected_validation_sha256,
)

assert manifest.get("split_version") == "silver-split-1.0.0"
assert manifest.get("test_is_locked") is True
assert validation_entry["path"] == "validation_silver_v1.jsonl"
assert (
    validation_sha256
    == "7c2f5f911ea33c6854ad1adc21e41a58"
       "befb6b24b31cb5ef2b8e03b7b771477c"
)
assert validation_sha256 == expected_validation_sha256

print("\nManifest dan validation sesuai dengan frozen split A6.")

Split version: silver-split-1.0.0
Test dikunci: True
Validation path dalam manifest: validation_silver_v1.jsonl
Manifest SHA-256: 2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0
Validation SHA-256 aktual: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c
Validation SHA-256 manifest: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c
Validation hash sesuai: True

Manifest dan validation sesuai dengan frozen split A6.


In [8]:
import shutil
from pathlib import Path

SPLIT_DIR = Path("/content/a8-splits")

assert not SPLIT_DIR.exists(), (
    f"{SPLIT_DIR} sudah ada. "
    "Jangan menimpa direktori input tanpa pemeriksaan."
)

SPLIT_DIR.mkdir(parents=False, exist_ok=False)

runtime_manifest = (
    SPLIT_DIR / "split_manifest_silver_v1.json"
)
runtime_validation = (
    SPLIT_DIR / "validation_silver_v1.jsonl"
)

shutil.copy2(MANIFEST_SOURCE, runtime_manifest)
shutil.copy2(VALIDATION_SOURCE, runtime_validation)

runtime_files = sorted(
    path.name
    for path in SPLIT_DIR.iterdir()
    if path.is_file()
)

print("File runtime Phase 1:", runtime_files)
print(
    "Manifest runtime SHA-256:",
    sha256_file(runtime_manifest),
)
print(
    "Validation runtime SHA-256:",
    sha256_file(runtime_validation),
)

assert runtime_files == [
    "split_manifest_silver_v1.json",
    "validation_silver_v1.jsonl",
]
assert sha256_file(runtime_manifest) == manifest_sha256
assert sha256_file(runtime_validation) == validation_sha256
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()
assert not (SPLIT_DIR / "train_silver_v1.jsonl").exists()

print("\nRuntime Phase 1 hanya berisi manifest dan validation.")
print("Locked test tidak tersedia di runtime.")

File runtime Phase 1: ['split_manifest_silver_v1.json', 'validation_silver_v1.jsonl']
Manifest runtime SHA-256: 2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0
Validation runtime SHA-256: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c

Runtime Phase 1 hanya berisi manifest dan validation.
Locked test tidak tersedia di runtime.


In [9]:
import json
from pathlib import Path

from sipature_ml.config import load_config

training_config = load_config("training")
taxonomy_config = load_config("taxonomy")

indobert_config = training_config["indobert"]
calibration_config = training_config["calibration"]
aspect_labels = sorted(taxonomy_config["aspect_definitions"])

print("Model ID:", indobert_config["model_id"])
print("Model revision:", indobert_config["revision"])
print("Max length:", indobert_config["max_length"])
print("Calibration version:", calibration_config["version"])
print("Output version:", calibration_config["output_version"])
print("Calibration method:", calibration_config["method"])
print(
    "Temperature range:",
    calibration_config["temperature_min"],
    "sampai",
    calibration_config["temperature_max"],
)
print(
    "Temperature grid size:",
    calibration_config["temperature_grid_size"],
)
print("ECE bins:", calibration_config["ece_bins"])
print(
    "Jumlah threshold candidates:",
    len(calibration_config["threshold_candidates"]),
)
print(
    "Alert minimum precision:",
    calibration_config["alert_minimum_precision"],
)
print(
    "Alert minimum predictions:",
    calibration_config["alert_minimum_predictions"],
)
print(
    "Inference batch size:",
    calibration_config["inference_batch_size"],
)
print("Require CUDA:", calibration_config["require_cuda"])
print("Jumlah aspek:", len(aspect_labels))
print("Urutan aspek:", aspect_labels)

assert indobert_config["model_id"] == (
    "indobenchmark/indobert-base-p1"
)
assert indobert_config["revision"] == (
    "c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2"
)
assert indobert_config["max_length"] == 192

assert calibration_config["version"] == (
    "aspect-calibration-v1"
)
assert calibration_config["output_version"] == (
    "indobert-evaluation-v1"
)
assert calibration_config["method"] == (
    "deterministic_bounded_temperature_grid"
)
assert calibration_config["temperature_min"] == 0.5
assert calibration_config["temperature_max"] == 5.0
assert calibration_config["temperature_grid_size"] == 181
assert calibration_config["ece_bins"] == 10
assert calibration_config["alert_minimum_precision"] == 0.80
assert calibration_config["alert_minimum_predictions"] == 5
assert calibration_config["inference_batch_size"] == 32
assert calibration_config["require_cuda"] is True
assert len(aspect_labels) == 14

print("\nKonfigurasi IndoBERT, calibration, dan taxonomy sesuai.")

Model ID: indobenchmark/indobert-base-p1
Model revision: c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2
Max length: 192
Calibration version: aspect-calibration-v1
Output version: indobert-evaluation-v1
Calibration method: deterministic_bounded_temperature_grid
Temperature range: 0.5 sampai 5.0
Temperature grid size: 181
ECE bins: 10
Jumlah threshold candidates: 19
Alert minimum precision: 0.8
Alert minimum predictions: 5
Inference batch size: 32
Require CUDA: True
Jumlah aspek: 14
Urutan aspek: ['access', 'cleanliness', 'comfort', 'crowding', 'maintenance', 'opening_hours', 'parking', 'price_transparency', 'public_facilities', 'safety', 'sanitation', 'scenery', 'staff_service', 'waste']

Konfigurasi IndoBERT, calibration, dan taxonomy sesuai.


In [10]:
import json
import time

from sipature_ml.evaluation import validate_model_contract

a7_manifest_path = MODEL_RUN_DIR / "manifest.json"

assert a7_manifest_path.is_file()

a7_manifest = json.loads(
    a7_manifest_path.read_text(encoding="utf-8")
)

print("Mulai verifikasi artifact model A7...")
print("Sekitar 1 GB bobot akan dibaca dari Google Drive.")

started_at = time.perf_counter()

verified_model_hashes = validate_model_contract(
    MODEL_RUN_DIR,
    a7_manifest,
)

elapsed_seconds = time.perf_counter() - started_at

print(
    "\nJumlah artifact model terverifikasi:",
    len(verified_model_hashes),
)

for relative_path, digest in sorted(
    verified_model_hashes.items()
):
    print(f"- {relative_path}")
    print(f"  SHA-256: {digest}")

print(
    "\nWaktu verifikasi:",
    round(elapsed_seconds, 2),
    "detik",
)

required_reload_files = {
    "aspect/model/config.json",
    "aspect/model/model.safetensors",
    "aspect/model/special_tokens_map.json",
    "aspect/model/tokenizer_config.json",
    "aspect/model/vocab.txt",
    "polarity/model/config.json",
    "polarity/model/model.safetensors",
    "polarity/model/special_tokens_map.json",
    "polarity/model/tokenizer_config.json",
    "polarity/model/vocab.txt",
}

missing_verified_files = (
    required_reload_files - set(verified_model_hashes)
)

print(
    "Artifact wajib yang belum terverifikasi:",
    sorted(missing_verified_files),
)

assert not missing_verified_files

print(
    "\nSeluruh artifact local reload A7 "
    "cocok dengan manifest."
)

Mulai verifikasi artifact model A7...
Sekitar 1 GB bobot akan dibaca dari Google Drive.

Jumlah artifact model terverifikasi: 10
- aspect/model/config.json
  SHA-256: 953072ce4625fa8c32c1022904a1c9a18f15836e670b5a48fea7c7527b5f2557
- aspect/model/model.safetensors
  SHA-256: 52906dad616f585710fddc576a922c2efa6a10d813ac68af8444431e3f8df4d9
- aspect/model/special_tokens_map.json
  SHA-256: b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3
- aspect/model/tokenizer_config.json
  SHA-256: 0088a6f8bcdd4014184fb068b83ebb12896a9db2bb269a71f73de83fef24bceb
- aspect/model/vocab.txt
  SHA-256: 35cfc7be6dc3ec71102248722f352830a8fd89b5a5dadb83d70c2f7794ddbc11
- polarity/model/config.json
  SHA-256: afa61f4fb24db810ec38bf2d41cfc3c3a0267219ab9fd3a35f9275671e3c419a
- polarity/model/model.safetensors
  SHA-256: de30bf7f8f5dbb6b8faab20d9840b1fc5e1720dcb1c3b4e96aa4417795740f72
- polarity/model/special_tokens_map.json
  SHA-256: b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237e

In [11]:
from pathlib import Path

CALIBRATION_ID = (
    "20260801_indobert-silver-v1_calibration-v1"
)

CALIBRATION_DIR = (
    Path("/content/drive/MyDrive/SIPATURE/calibration")
    / CALIBRATION_ID
)

print("Calibration ID:", CALIBRATION_ID)
print("Calibration directory:", CALIBRATION_DIR)
print(
    "Direktori sudah ada:",
    CALIBRATION_DIR.exists(),
)

assert not CALIBRATION_DIR.exists(), (
    "Direktori kalibrasi sudah ada. "
    "Jangan menimpa artifact immutable. "
    "Berhenti dan periksa direktori tersebut."
)

assert SPLIT_DIR == Path("/content/a8-splits")
assert MODEL_RUN_DIR.name == (
    "20260801-1024_indobert-silver-v1"
)

runtime_files = sorted(
    path.name
    for path in SPLIT_DIR.iterdir()
    if path.is_file()
)

print("File runtime:", runtime_files)

assert runtime_files == [
    "split_manifest_silver_v1.json",
    "validation_silver_v1.jsonl",
]
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()

print("\nID kalibrasi tersedia.")
print("Locked test tetap tidak tersedia di runtime.")

Calibration ID: 20260801_indobert-silver-v1_calibration-v1
Calibration directory: /content/drive/MyDrive/SIPATURE/calibration/20260801_indobert-silver-v1_calibration-v1
Direktori sudah ada: False
File runtime: ['split_manifest_silver_v1.json', 'validation_silver_v1.jsonl']

ID kalibrasi tersedia.
Locked test tetap tidak tersedia di runtime.


In [12]:
import json
import time

import torch

from sipature_ml.evaluation import run_calibration

assert torch.cuda.is_available()
assert torch.cuda.get_device_name(0) == "Tesla T4"
assert not CALIBRATION_DIR.exists()
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()

print("Memulai validation-only calibration...")
print("GPU:", torch.cuda.get_device_name(0))
print("Split directory:", SPLIT_DIR)
print("Model run:", MODEL_RUN_DIR)
print("Calibration output:", CALIBRATION_DIR)
print("Locked test tersedia:", (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists())

started_at = time.perf_counter()

calibration = run_calibration(
    split_dir=SPLIT_DIR,
    model_run_dir=MODEL_RUN_DIR,
    output_dir=CALIBRATION_DIR,
)

elapsed_seconds = time.perf_counter() - started_at

print("\nCalibration selesai.")
print("Waktu total:", round(elapsed_seconds, 2), "detik")
print("Phase:", calibration["phase"])
print("Test dibaca:", calibration["test_read"])
print("Device:", calibration["inference"]["device"])
print(
    "Model loading:",
    round(
        calibration["inference"]["model_loading_seconds"],
        4,
    ),
    "detik",
)
print(
    "Aspect inference:",
    round(
        calibration["inference"]["aspect_latency_seconds"],
        4,
    ),
    "detik",
)
print(
    "Polarity inference:",
    round(
        calibration["inference"]["polarity_latency_seconds"],
        4,
    ),
    "detik",
)

validation_metrics = calibration["validation_calibration"]

print("\nTemperature:", calibration["temperature"])
print("NLL sebelum:", validation_metrics["nll_before"])
print("NLL sesudah:", validation_metrics["nll_after"])
print("ECE sebelum:", validation_metrics["ece_before"])
print("ECE sesudah:", validation_metrics["ece_after"])
print("Brier sebelum:", validation_metrics["brier_before"])
print("Brier sesudah:", validation_metrics["brier_after"])

print(
    "\nValidation polarity Macro F1:",
    calibration["validation_polarity"]["macro_f1"],
)
print(
    "Validation polarity support:",
    calibration["validation_polarity"]["support"],
)

print("\nDetection thresholds:")
for aspect, threshold in calibration[
    "detection_thresholds"
].items():
    print(f"- {aspect}: {threshold}")

print("\nAlert thresholds:")
for aspect in calibration["labels"]:
    alert = calibration["alert_validation"][aspect]
    print(
        f"- {aspect}: "
        f"threshold={alert['threshold']}, "
        f"precision={alert['precision']}, "
        f"recall={alert['recall']}, "
        f"predicted_support={alert['predicted_support']}, "
        f"target_met={alert['target_met']}"
    )

assert calibration["phase"] == "validation_calibration"
assert calibration["test_read"] is False
assert calibration["inference"]["device"] == "cuda"
assert CALIBRATION_DIR.is_dir()
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()

print("\nVALIDATION CALIBRATION BERHASIL.")
print("LOCKED TEST TIDAK DIBACA.")
print("BERHENTI SEBELUM MANDATORY CONFIRMATION.")

Memulai validation-only calibration...
GPU: Tesla T4
Split directory: /content/a8-splits
Model run: /content/drive/MyDrive/SIPATURE/runs/20260801-1024_indobert-silver-v1
Calibration output: /content/drive/MyDrive/SIPATURE/calibration/20260801_indobert-silver-v1_calibration-v1
Locked test tersedia: False

Calibration selesai.
Waktu total: 28.36 detik
Phase: validation_calibration
Test dibaca: False
Device: cuda
Model loading: 1.8261 detik
Aspect inference: 2.1584 detik
Polarity inference: 2.2786 detik

Temperature: 0.6
NLL sebelum: 0.4533156870788427
NLL sesudah: 0.4235609447556627
ECE sebelum: 0.2705768614228283
ECE sesudah: 0.22529126900754612
Brier sebelum: 0.1440880331928714
Brier sesudah: 0.13877199324016595

Validation polarity Macro F1: 0.7044036969463576
Validation polarity support: 260

Detection thresholds:
- access: 0.75
- cleanliness: 0.75
- comfort: 0.65
- crowding: 0.8
- maintenance: 0.85
- opening_hours: 0.55
- parking: 0.9
- price_transparency: 0.7
- public_facilities: 0

In [13]:
import hashlib
import json
from pathlib import Path

def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

required_calibration_files = [
    "calibration.json",
    "manifest.json",
    "validation-predictions.npz",
    "calibration.png",
]

print("Calibration directory:", CALIBRATION_DIR)
print("Ditemukan:", CALIBRATION_DIR.is_dir())

assert CALIBRATION_DIR.is_dir()

missing_files = []

for filename in required_calibration_files:
    path = CALIBRATION_DIR / filename
    exists = path.is_file()
    size = path.stat().st_size if exists else None

    print(
        f"{filename}:",
        "ADA" if exists else "HILANG",
        f"({size:,} byte)" if size is not None else "",
    )

    if not exists:
        missing_files.append(filename)

assert not missing_files, (
    "Artifact kalibrasi tidak lengkap: "
    + ", ".join(missing_files)
)

calibration_path = CALIBRATION_DIR / "calibration.json"
calibration_manifest_path = (
    CALIBRATION_DIR / "manifest.json"
)

frozen_calibration = json.loads(
    calibration_path.read_text(encoding="utf-8")
)
calibration_manifest = json.loads(
    calibration_manifest_path.read_text(encoding="utf-8")
)

calibration_sha256 = sha256_stream(calibration_path)
calibration_manifest_sha256 = sha256_stream(
    calibration_manifest_path
)

print("\nCalibration SHA-256:", calibration_sha256)
print(
    "Calibration SHA-256 dalam manifest:",
    calibration_manifest.get("calibration_sha256"),
)
print(
    "Calibration hash sesuai:",
    calibration_sha256
    == calibration_manifest.get("calibration_sha256"),
)
print(
    "Manifest SHA-256:",
    calibration_manifest_sha256,
)
print(
    "Manifest phase:",
    calibration_manifest.get("phase"),
)
print(
    "Manifest test_read:",
    calibration_manifest.get("test_read"),
)

assert (
    calibration_sha256
    == calibration_manifest["calibration_sha256"]
)
assert (
    calibration_manifest["phase"]
    == "validation_calibration"
)
assert calibration_manifest["test_read"] is False
assert frozen_calibration["test_read"] is False

print("\nVerifikasi artifact hashes:")

artifact_hashes = calibration_manifest.get(
    "artifact_hashes",
    {},
)

assert artifact_hashes

for relative_path, expected_hash in sorted(
    artifact_hashes.items()
):
    artifact_path = CALIBRATION_DIR / relative_path

    assert artifact_path.is_file(), (
        f"Artifact hilang: {relative_path}"
    )

    actual_hash = sha256_stream(artifact_path)
    matches = actual_hash == expected_hash

    print(
        f"- {relative_path}:",
        "COCOK" if matches else "TIDAK COCOK",
    )
    print("  SHA-256:", actual_hash)

    assert matches, (
        f"Hash artifact tidak cocok: {relative_path}"
    )

print("\nSeluruh artifact kalibrasi cocok dengan manifest.")

Calibration directory: /content/drive/MyDrive/SIPATURE/calibration/20260801_indobert-silver-v1_calibration-v1
Ditemukan: True
calibration.json: ADA (7,006 byte)
manifest.json: ADA (466 byte)
validation-predictions.npz: ADA (37,386 byte)
calibration.png: ADA (31,535 byte)

Calibration SHA-256: f7efc6525c2b7de9351b31257beb7455db7f3b1c7abe675a412038505123eadb
Calibration SHA-256 dalam manifest: f7efc6525c2b7de9351b31257beb7455db7f3b1c7abe675a412038505123eadb
Calibration hash sesuai: True
Manifest SHA-256: 0fb30bf3170a62bc4a7e257d9fcf9393b42c78eee7239e80aa1c435dff878e3a
Manifest phase: validation_calibration
Manifest test_read: False

Verifikasi artifact hashes:
- calibration.json: COCOK
  SHA-256: f7efc6525c2b7de9351b31257beb7455db7f3b1c7abe675a412038505123eadb
- calibration.png: COCOK
  SHA-256: 16960431e85b9f15212e7d2a3cec2763ee2f771a428a4a3a49f0f26895fa6a99
- validation-predictions.npz: COCOK
  SHA-256: 127c800b0597484e2a209537fc9c56de34644efbba6b8e4fca9fb151e085949c

Seluruh artifact 

In [14]:
print("Phase:", frozen_calibration["phase"])
print("Test dibaca:", frozen_calibration["test_read"])
print(
    "Validation SHA-256:",
    frozen_calibration["validation_sha256"],
)
print(
    "Split manifest SHA-256:",
    frozen_calibration["split_manifest_sha256"],
)
print(
    "A7 manifest SHA-256:",
    frozen_calibration["a7_manifest_sha256"],
)
print(
    "Current training config SHA-256:",
    frozen_calibration[
        "current_training_config_sha256"
    ],
)
print(
    "IndoBERT config canonical SHA-256:",
    frozen_calibration[
        "indobert_config_canonical_sha256"
    ],
)
print(
    "Calibration config canonical SHA-256:",
    frozen_calibration[
        "calibration_config_canonical_sha256"
    ],
)
print(
    "Jumlah model hashes:",
    len(frozen_calibration["model_hashes"]),
)
print(
    "Jumlah labels:",
    len(frozen_calibration["labels"]),
)

assert frozen_calibration["phase"] == (
    "validation_calibration"
)
assert frozen_calibration["test_read"] is False
assert frozen_calibration["validation_sha256"] == (
    "7c2f5f911ea33c6854ad1adc21e41a58"
    "befb6b24b31cb5ef2b8e03b7b771477c"
)
assert frozen_calibration["split_manifest_sha256"] == (
    "2044659130e0f2a80bc6c15bd75052ec"
    "ce51c8a064f8c24ef8eb86d06110a2c0"
)
assert frozen_calibration["labels"] == aspect_labels
assert len(frozen_calibration["labels"]) == 14
assert len(frozen_calibration["model_hashes"]) == 10
assert frozen_calibration["temperature"] == 0.6

print("\nKontrak pembekuan kalibrasi sesuai.")

Phase: validation_calibration
Test dibaca: False
Validation SHA-256: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c
Split manifest SHA-256: 2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0
A7 manifest SHA-256: e9e2d8e5676a94a6b5ebcd01d4ac5a512bfa042ed90dc615bd480abf7f630d01
Current training config SHA-256: 2dd885607b83313c61276f650b189125f52aa1a8d9b278c27fde5d8c81a053ea
IndoBERT config canonical SHA-256: 7e3b2a62ac6e8cdccb75152f0271c762a01a5f9d9f9c2583b84c8b5aa842bd95
Calibration config canonical SHA-256: 4531b2c101900450e3cc245934eee00725e29c2c6647746b9c2ad2fa084cf0ff
Jumlah model hashes: 10
Jumlah labels: 14

Kontrak pembekuan kalibrasi sesuai.


In [15]:
import numpy as np

from sipature_ml.evaluation import aspect_metrics

prediction_archive = np.load(
    CALIBRATION_DIR / "validation-predictions.npz",
    allow_pickle=False,
)

print(
    "Isi prediction archive:",
    prediction_archive.files,
)

validation_probabilities = prediction_archive[
    "probabilities"
]
validation_targets = prediction_archive["targets"]
validation_logits = prediction_archive["logits"]
validation_polarity_logits = prediction_archive[
    "polarity_logits"
]

detection_thresholds = [
    frozen_calibration["detection_thresholds"][label]
    for label in frozen_calibration["labels"]
]

alert_thresholds = [
    frozen_calibration["alert_thresholds"][label]
    for label in frozen_calibration["labels"]
]

tuned_validation_metrics = aspect_metrics(
    targets=validation_targets,
    probabilities=validation_probabilities,
    thresholds=detection_thresholds,
    labels=frozen_calibration["labels"],
    alert_thresholds=alert_thresholds,
)

temporary_validation_metrics = aspect_metrics(
    targets=validation_targets,
    probabilities=validation_probabilities,
    thresholds=[0.5] * len(frozen_calibration["labels"]),
    labels=frozen_calibration["labels"],
)

print(
    "Aspect logits shape:",
    validation_logits.shape,
)
print(
    "Aspect targets shape:",
    validation_targets.shape,
)
print(
    "Polarity logits shape:",
    validation_polarity_logits.shape,
)

print("\nThreshold sementara 0.50:")
print(
    "Macro F1:",
    temporary_validation_metrics["macro_f1"],
)
print(
    "Micro F1:",
    temporary_validation_metrics["micro_f1"],
)

print("\nThreshold final per aspek:")
print(
    "Macro F1:",
    tuned_validation_metrics["macro_f1"],
)
print(
    "Micro F1:",
    tuned_validation_metrics["micro_f1"],
)

macro_improvement = (
    tuned_validation_metrics["macro_f1"]
    - temporary_validation_metrics["macro_f1"]
)
micro_improvement = (
    tuned_validation_metrics["micro_f1"]
    - temporary_validation_metrics["micro_f1"]
)

print(
    "Perubahan Macro F1:",
    macro_improvement,
)
print(
    "Perubahan Micro F1:",
    micro_improvement,
)

print("\nPer-label validation metrics:")

for label in frozen_calibration["labels"]:
    result = tuned_validation_metrics["per_label"][label]

    print(
        f"- {label}: "
        f"F1={result['f1']:.6f}, "
        f"precision={result['precision']:.6f}, "
        f"recall={result['recall']:.6f}, "
        f"support={result['support']}, "
        f"predicted_support="
        f"{result['predicted_support']}"
    )

assert validation_logits.shape == (196, 14)
assert validation_targets.shape == (196, 14)
assert (
    tuned_validation_metrics["macro_f1"]
    >= temporary_validation_metrics["macro_f1"]
)

print("\nThreshold final tidak memperburuk validation Macro F1.")

Isi prediction archive: ['logits', 'targets', 'probabilities', 'polarity_logits']
Aspect logits shape: (196, 14)
Aspect targets shape: (196, 14)
Polarity logits shape: (260, 3)

Threshold sementara 0.50:
Macro F1: 0.4012246461923619
Micro F1: 0.4326450344149459

Threshold final per aspek:
Macro F1: 0.553546118292758
Micro F1: 0.56957928802589
Perubahan Macro F1: 0.1523214721003961
Perubahan Micro F1: 0.13693425361094408

Per-label validation metrics:
- access: F1=0.528302, precision=0.424242, recall=0.700000, support=20, predicted_support=33
- cleanliness: F1=0.625000, precision=0.571429, recall=0.689655, support=29, predicted_support=35
- comfort: F1=0.394366, precision=0.274510, recall=0.700000, support=20, predicted_support=51
- crowding: F1=0.428571, precision=0.375000, recall=0.500000, support=6, predicted_support=8
- maintenance: F1=0.600000, precision=0.562500, recall=0.642857, support=14, predicted_support=16
- opening_hours: F1=0.352941, precision=0.214286, recall=1.000000, su

In [16]:
alert_summary = []

for label in frozen_calibration["labels"]:
    alert = frozen_calibration["alert_validation"][label]

    alert_summary.append(
        {
            "aspect": label,
            "threshold": alert["threshold"],
            "precision": alert["precision"],
            "recall": alert["recall"],
            "predicted_support": (
                alert["predicted_support"]
            ),
            "target_met": alert["target_met"],
        }
    )

target_met_labels = [
    row["aspect"]
    for row in alert_summary
    if row["target_met"]
]

target_not_met_labels = [
    row["aspect"]
    for row in alert_summary
    if not row["target_met"]
]

print(
    "Alert target tercapai:",
    len(target_met_labels),
    "dari",
    len(alert_summary),
)
print("Aspek target tercapai:", target_met_labels)
print(
    "Aspek target belum tercapai:",
    target_not_met_labels,
)

assert target_met_labels == [
    "cleanliness",
    "parking",
    "scenery",
    "staff_service",
    "waste",
]

print(
    "\nAlert untuk aspek target_not_met harus dibaca "
    "sebagai threshold kandidat, bukan high-precision alert."
)

Alert target tercapai: 5 dari 14
Aspek target tercapai: ['cleanliness', 'parking', 'scenery', 'staff_service', 'waste']
Aspek target belum tercapai: ['access', 'comfort', 'crowding', 'maintenance', 'opening_hours', 'price_transparency', 'public_facilities', 'safety', 'sanitation']

Alert untuk aspek target_not_met harus dibaca sebagai threshold kandidat, bukan high-precision alert.


In [17]:
import json
from datetime import datetime, timezone
from pathlib import Path

FREEZE_RECEIPT = (
    CALIBRATION_DIR.parent
    / f"{CALIBRATION_ID}_freeze-receipt.json"
)

assert not FREEZE_RECEIPT.exists(), (
    "Freeze receipt sudah ada. Jangan menimpanya."
)

freeze_receipt = {
    "calibration_id": CALIBRATION_ID,
    "frozen_at": datetime.now(timezone.utc).isoformat(),
    "status": "accepted_and_frozen_before_locked_test",
    "test_read": False,
    "calibration_directory": str(CALIBRATION_DIR),
    "calibration_sha256": calibration_sha256,
    "calibration_manifest_sha256": (
        calibration_manifest_sha256
    ),
    "validation_sha256": (
        frozen_calibration["validation_sha256"]
    ),
    "split_manifest_sha256": (
        frozen_calibration["split_manifest_sha256"]
    ),
    "a7_manifest_sha256": (
        frozen_calibration["a7_manifest_sha256"]
    ),
    "current_training_config_sha256": (
        frozen_calibration[
            "current_training_config_sha256"
        ]
    ),
    "indobert_config_canonical_sha256": (
        frozen_calibration[
            "indobert_config_canonical_sha256"
        ]
    ),
    "calibration_config_canonical_sha256": (
        frozen_calibration[
            "calibration_config_canonical_sha256"
        ]
    ),
    "temperature": frozen_calibration["temperature"],
    "validation_metrics": {
        "macro_f1_temporary_threshold_0_50": (
            temporary_validation_metrics["macro_f1"]
        ),
        "micro_f1_temporary_threshold_0_50": (
            temporary_validation_metrics["micro_f1"]
        ),
        "macro_f1_tuned_thresholds": (
            tuned_validation_metrics["macro_f1"]
        ),
        "micro_f1_tuned_thresholds": (
            tuned_validation_metrics["micro_f1"]
        ),
        "nll_before": frozen_calibration[
            "validation_calibration"
        ]["nll_before"],
        "nll_after": frozen_calibration[
            "validation_calibration"
        ]["nll_after"],
        "ece_before": frozen_calibration[
            "validation_calibration"
        ]["ece_before"],
        "ece_after": frozen_calibration[
            "validation_calibration"
        ]["ece_after"],
        "brier_before": frozen_calibration[
            "validation_calibration"
        ]["brier_before"],
        "brier_after": frozen_calibration[
            "validation_calibration"
        ]["brier_after"],
        "polarity_macro_f1": frozen_calibration[
            "validation_polarity"
        ]["macro_f1"],
    },
    "alert_target_met_labels": target_met_labels,
    "alert_target_not_met_labels": target_not_met_labels,
    "locked_test_policy": (
        "No model, temperature, threshold, taxonomy, or "
        "configuration changes are allowed after this receipt."
    ),
}

FREEZE_RECEIPT.write_text(
    json.dumps(
        freeze_receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

freeze_receipt_sha256 = sha256_stream(FREEZE_RECEIPT)

print("Freeze receipt:", FREEZE_RECEIPT)
print("Freeze receipt SHA-256:", freeze_receipt_sha256)
print("Status:", freeze_receipt["status"])
print("Test dibaca:", freeze_receipt["test_read"])

assert freeze_receipt["test_read"] is False
assert freeze_receipt["temperature"] == 0.6
assert (
    freeze_receipt["calibration_sha256"]
    == calibration_sha256
)

print("\nKALIBRASI DITERIMA DAN DIBEKUKAN.")
print("LOCKED TEST MASIH BELUM DIAKSES.")

Freeze receipt: /content/drive/MyDrive/SIPATURE/calibration/20260801_indobert-silver-v1_calibration-v1_freeze-receipt.json
Freeze receipt SHA-256: e3ab195f6907c7904b6516109947ca4369270c948297ee97ffdf3b415268b393
Status: accepted_and_frozen_before_locked_test
Test dibaca: False

KALIBRASI DITERIMA DAN DIBEKUKAN.
LOCKED TEST MASIH BELUM DIAKSES.


In [18]:
post_freeze_calibration_sha256 = sha256_stream(
    CALIBRATION_DIR / "calibration.json"
)
post_freeze_manifest_sha256 = sha256_stream(
    CALIBRATION_DIR / "manifest.json"
)

print(
    "Calibration hash tetap:",
    post_freeze_calibration_sha256
    == calibration_sha256,
)
print(
    "Manifest hash tetap:",
    post_freeze_manifest_sha256
    == calibration_manifest_sha256,
)
print(
    "Test ada di runtime:",
    (SPLIT_DIR / "test_silver_v1.jsonl").exists(),
)

assert (
    post_freeze_calibration_sha256
    == calibration_sha256
)
assert (
    post_freeze_manifest_sha256
    == calibration_manifest_sha256
)
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nArtifact frozen tidak berubah.")

Calibration hash tetap: True
Manifest hash tetap: True
Test ada di runtime: False

Artifact frozen tidak berubah.


In [20]:
from pathlib import Path

LOCKED_TEST_SOURCE = Path(
    "/content/drive/MyDrive/SIPATURE/locked-test/"
    "test_silver_v1.jsonl"
)

print("Locked test source:", LOCKED_TEST_SOURCE)
print("File ditemukan:", LOCKED_TEST_SOURCE.is_file())

if LOCKED_TEST_SOURCE.is_file():
    print(
        "Ukuran file:",
        f"{LOCKED_TEST_SOURCE.stat().st_size:,}",
        "byte",
    )

print(
    "Test ada di runtime Phase 1:",
    (SPLIT_DIR / "test_silver_v1.jsonl").exists(),
)

assert LOCKED_TEST_SOURCE.is_file()
assert LOCKED_TEST_SOURCE.name == "test_silver_v1.jsonl"
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nLocked test tersedia di penyimpanan terkontrol.")
print("Locked test belum disalin ke runtime.")
print("Isi dan hash locked test belum dibaca di Colab.")

Locked test source: /content/drive/MyDrive/SIPATURE/locked-test/test_silver_v1.jsonl
File ditemukan: True
Ukuran file: 164,195 byte
Test ada di runtime Phase 1: False

Locked test tersedia di penyimpanan terkontrol.
Locked test belum disalin ke runtime.
Isi dan hash locked test belum dibaca di Colab.


In [21]:
from pathlib import Path

EVALUATION_ID = (
    "20260801_indobert-silver-v1_locked-test-v1"
)

EVALUATION_DIR = (
    Path("/content/drive/MyDrive/SIPATURE/evaluation")
    / EVALUATION_ID
)

BASELINE_METRICS_DIR = Path(
    "/content/hackathon/ml/artifacts/metrics"
)

print("Evaluation ID:", EVALUATION_ID)
print("Evaluation directory:", EVALUATION_DIR)
print(
    "Direktori evaluasi sudah ada:",
    EVALUATION_DIR.exists(),
)
print(
    "Baseline metrics directory:",
    BASELINE_METRICS_DIR,
)
print(
    "Baseline directory tersedia:",
    BASELINE_METRICS_DIR.is_dir(),
)
print(
    "Locked test ada di runtime:",
    (SPLIT_DIR / "test_silver_v1.jsonl").exists(),
)

assert not EVALUATION_DIR.exists(), (
    "Evaluation directory sudah ada. "
    "Jangan menghapus, menimpa, atau mengganti ID. "
    "Berhenti untuk pemeriksaan."
)
assert BASELINE_METRICS_DIR.is_dir()
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nEvaluation ID tersedia dan belum diklaim.")
print("Locked test tetap di penyimpanan terkontrol.")

Evaluation ID: 20260801_indobert-silver-v1_locked-test-v1
Evaluation directory: /content/drive/MyDrive/SIPATURE/evaluation/20260801_indobert-silver-v1_locked-test-v1
Direktori evaluasi sudah ada: False
Baseline metrics directory: /content/hackathon/ml/artifacts/metrics
Baseline directory tersedia: True
Locked test ada di runtime: False

Evaluation ID tersedia dan belum diklaim.
Locked test tetap di penyimpanan terkontrol.


In [23]:
from pathlib import Path

BASELINE_METRICS_DIR = Path(
    "/content/drive/MyDrive/SIPATURE/baseline-metrics"
)

baseline_paths = {
    "keyword": (
        BASELINE_METRICS_DIR
        / "keyword-silver-v1-test-metrics.json"
    ),
    "tfidf": (
        BASELINE_METRICS_DIR
        / "tfidf-silver-v1-test-metrics.json"
    ),
}

print("Baseline directory:", BASELINE_METRICS_DIR)
print(
    "Baseline directory tersedia:",
    BASELINE_METRICS_DIR.is_dir(),
)

for name, path in baseline_paths.items():
    print(
        f"{name}:",
        "ADA" if path.is_file() else "HILANG",
        f"({path.stat().st_size:,} byte)"
        if path.is_file()
        else "",
    )

assert BASELINE_METRICS_DIR.is_dir()
assert all(
    path.is_file()
    for path in baseline_paths.values()
)

assert not EVALUATION_DIR.exists()
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nKedua baseline metrics tersedia.")
print("Evaluation directory tetap belum dibuat.")
print("Locked test tetap belum ada di runtime.")

Baseline directory: /content/drive/MyDrive/SIPATURE/baseline-metrics
Baseline directory tersedia: True
keyword: ADA (5,033 byte)
tfidf: ADA (6,734 byte)

Kedua baseline metrics tersedia.
Evaluation directory tetap belum dibuat.
Locked test tetap belum ada di runtime.


In [24]:
import json

baseline_summary = {}

for name, path in baseline_paths.items():
    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    baseline_summary[name] = {
        "model_version": payload.get("model_version"),
        "macro_f1": payload.get("macro_f1"),
        "micro_f1": payload.get("micro_f1"),
        "records": payload.get("records"),
        "reference_label_type": payload.get(
            "reference_label_type"
        ),
        "split_version": payload.get("split_version"),
        "split_manifest_sha256": payload.get(
            "split_manifest_sha256"
        ),
    }

    print(f"\n{name.upper()}:")
    print(
        json.dumps(
            baseline_summary[name],
            indent=2,
            sort_keys=True,
        )
    )

assert (
    baseline_summary["keyword"]["model_version"]
    == "keyword-silver-v1"
)
assert (
    baseline_summary["tfidf"]["model_version"]
    == "tfidf-aspect-silver-v1"
)
assert baseline_summary["keyword"]["records"] == 202
assert baseline_summary["tfidf"]["records"] == 202

assert baseline_summary["keyword"]["macro_f1"] == (
    0.9767532595118802
)
assert baseline_summary["keyword"]["micro_f1"] == (
    0.9783037475345168
)
assert baseline_summary["tfidf"]["macro_f1"] == (
    0.7200524598653073
)
assert baseline_summary["tfidf"]["micro_f1"] == (
    0.804040404040404
)

print("\nBaseline metrics sesuai hasil A6.")
print("Locked test belum diakses.")


KEYWORD:
{
  "macro_f1": 0.9767532595118802,
  "micro_f1": 0.9783037475345168,
  "model_version": "keyword-silver-v1",
  "records": 202,
  "reference_label_type": "ai_assisted_weak_supervision_silver",
  "split_manifest_sha256": null,
  "split_version": null
}

TFIDF:
{
  "macro_f1": 0.7200524598653073,
  "micro_f1": 0.804040404040404,
  "model_version": "tfidf-aspect-silver-v1",
  "records": 202,
  "reference_label_type": "ai_assisted_weak_supervision_silver",
  "split_manifest_sha256": null,
  "split_version": null
}

Baseline metrics sesuai hasil A6.
Locked test belum diakses.


In [25]:
baseline_hashes = {
    name: sha256_stream(path)
    for name, path in baseline_paths.items()
}

print("Baseline SHA-256:")

for name, digest in baseline_hashes.items():
    print(f"- {name}: {digest}")

assert not EVALUATION_DIR.exists()
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nBaseline artifact telah dicatat.")
print("Evaluation ID tetap belum diklaim.")
print("Locked test tetap belum diakses.")

Baseline SHA-256:
- keyword: 4dceb869f37fee0dc6f3556f394b5d0b5f26f5585a7c6ac55c6f79fdadc4ca08
- tfidf: bc3daedc36d4c6b3c53a2f13750f6448dba5914a994a554846a2573d26462659

Baseline artifact telah dicatat.
Evaluation ID tetap belum diklaim.
Locked test tetap belum diakses.


In [26]:
import json
import time

from sipature_ml.config import load_config
from sipature_ml.evaluation import (
    _canonical_hash,
    _load_a7_contract,
    _load_baseline_metrics,
    _preflight_plotting,
    _resolve_calibration,
    _validate_artifact_map,
    _validate_baselines,
    validate_calibration_contract,
)
from sipature_ml.manifest import sha256_file

print("Memulai preflight non-test final...")
print("Locked test tidak akan disalin atau dibaca.")

assert BASELINE_METRICS_DIR == Path(
    "/content/drive/MyDrive/SIPATURE/baseline-metrics"
)
assert not EVALUATION_DIR.exists()
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

preflight_started = time.perf_counter()

config = load_config("training")
taxonomy = load_config("taxonomy")
labels = sorted(taxonomy["aspect_definitions"])

split_manifest_path = (
    SPLIT_DIR / "split_manifest_silver_v1.json"
)
split_manifest = json.loads(
    split_manifest_path.read_text(encoding="utf-8")
)
split_manifest_hash = sha256_file(
    split_manifest_path
)

assert split_manifest.get("test_is_locked") is True
assert split_manifest_hash == (
    frozen_calibration["split_manifest_sha256"]
)

print("1. Verifikasi frozen split manifest...")
print("   SHA-256:", split_manifest_hash)
print("   LULUS")

calibration_path, calibration_manifest_path = (
    _resolve_calibration(CALIBRATION_DIR)
)

preflight_calibration = json.loads(
    calibration_path.read_text(encoding="utf-8")
)
preflight_calibration_manifest = json.loads(
    calibration_manifest_path.read_text(
        encoding="utf-8"
    )
)
preflight_calibration_hash = sha256_file(
    calibration_path
)

print("2. Verifikasi seluruh artifact kalibrasi...")
_validate_artifact_map(
    calibration_path.parent,
    preflight_calibration_manifest,
)
print("   LULUS")

print("3. Verifikasi seluruh local reload artifact A7...")
a7_manifest, preflight_model_hashes = (
    _load_a7_contract(
        MODEL_RUN_DIR,
        config,
    )
)
print(
    "   LULUS:",
    len(preflight_model_hashes),
    "artifact",
)

validation_entry = (
    split_manifest.get("outputs", {}).get("validation")
)

assert isinstance(validation_entry, dict)
assert isinstance(
    validation_entry.get("sha256"),
    str,
)

print("4. Verifikasi kontrak frozen calibration...")
validate_calibration_contract(
    preflight_calibration,
    preflight_calibration_hash,
    preflight_calibration_manifest,
    preflight_model_hashes,
    split_manifest_hash,
    validation_entry["sha256"],
    labels,
    _canonical_hash(config["calibration"]),
    _canonical_hash(config["indobert"]),
)
print("   LULUS")

print("5. Verifikasi binding ke manifest A7...")
current_a7_manifest_hash = sha256_file(
    MODEL_RUN_DIR / "manifest.json"
)

assert (
    preflight_calibration["a7_manifest_sha256"]
    == current_a7_manifest_hash
)
print("   SHA-256:", current_a7_manifest_hash)
print("   LULUS")

print("6. Muat dan verifikasi baseline metrics...")
preflight_baseline = _load_baseline_metrics(
    BASELINE_METRICS_DIR
)
_validate_baselines(
    preflight_baseline,
    split_manifest,
    split_manifest_hash,
)

print(
    "   LULUS:",
    len(preflight_baseline["models"]),
    "baseline",
)

for model in preflight_baseline["models"]:
    print(
        "   -",
        model["model"],
        "Macro F1:",
        model["macro_f1"],
        "Micro F1:",
        model["micro_f1"],
    )

print("7. Verifikasi plotting dependency...")
_preflight_plotting()
print("   LULUS")

preflight_elapsed = (
    time.perf_counter() - preflight_started
)

print(
    "\nPreflight selesai dalam:",
    round(preflight_elapsed, 2),
    "detik",
)

print(
    "Split manifest SHA-256:",
    split_manifest_hash,
)
print(
    "Calibration SHA-256:",
    preflight_calibration_hash,
)
print(
    "A7 manifest SHA-256:",
    current_a7_manifest_hash,
)
print("Jumlah labels:", len(labels))
print(
    "Jumlah model hashes:",
    len(preflight_model_hashes),
)
print(
    "Jumlah baseline:",
    len(preflight_baseline["models"]),
)
print(
    "Evaluation directory ada:",
    EVALUATION_DIR.exists(),
)
print(
    "Locked test ada di runtime:",
    (SPLIT_DIR / "test_silver_v1.jsonl").exists(),
)

assert split_manifest_hash == (
    "2044659130e0f2a80bc6c15bd75052ec"
    "ce51c8a064f8c24ef8eb86d06110a2c0"
)
assert preflight_calibration_hash == (
    "f7efc6525c2b7de9351b31257beb7455"
    "db7f3b1c7abe675a412038505123eadb"
)
assert current_a7_manifest_hash == (
    "e9e2d8e5676a94a6b5ebcd01d4ac5a"
    "512bfa042ed90dc615bd480abf7f630d01"
)
assert len(labels) == 14
assert len(preflight_model_hashes) == 10
assert len(preflight_baseline["models"]) == 2
assert not EVALUATION_DIR.exists()
assert not (
    SPLIT_DIR / "test_silver_v1.jsonl"
).exists()

print("\nSELURUH PREFLIGHT NON-TEST LULUS.")
print("EVALUATION ID BELUM DIKLAIM.")
print("LOCKED TEST BELUM DIAKSES.")

Memulai preflight non-test final...
Locked test tidak akan disalin atau dibaca.
1. Verifikasi frozen split manifest...
   SHA-256: 2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0
   LULUS
2. Verifikasi seluruh artifact kalibrasi...
   LULUS
3. Verifikasi seluruh local reload artifact A7...
   LULUS: 10 artifact
4. Verifikasi kontrak frozen calibration...
   LULUS
5. Verifikasi binding ke manifest A7...
   SHA-256: e9e2d8e5676a94a6b5ebcd01d4ac5a512bfa042ed90dc615bd480abf7f630d01
   LULUS
6. Muat dan verifikasi baseline metrics...
   LULUS: 2 baseline
   - keyword-silver-v1 Macro F1: 0.9767532595118802 Micro F1: 0.9783037475345168
   - tfidf-aspect-silver-v1 Macro F1: 0.7200524598653073 Micro F1: 0.804040404040404
7. Verifikasi plotting dependency...
   LULUS

Preflight selesai dalam: 4.84 detik
Split manifest SHA-256: 2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0
Calibration SHA-256: f7efc6525c2b7de9351b31257beb7455db7f3b1c7abe675a412038505123eadb
A7 

In [28]:
import json
from pathlib import Path

state_path = (
    EVALUATION_DIR / "evaluation-state.json"
)
metrics_path = EVALUATION_DIR / "metrics.json"
manifest_path = EVALUATION_DIR / "manifest.json"

print("Evaluation directory:", EVALUATION_DIR)
print("State tersedia:", state_path.is_file())
print("Metrics tersedia:", metrics_path.is_file())
print("Manifest tersedia:", manifest_path.is_file())

assert EVALUATION_DIR.is_dir()
assert state_path.is_file()
assert metrics_path.is_file()
assert manifest_path.is_file()

state = json.loads(
    state_path.read_text(encoding="utf-8")
)
saved_metrics = json.loads(
    metrics_path.read_text(encoding="utf-8")
)
evaluation_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

print("\nEVALUATION STATE")
print(
    json.dumps(
        state,
        indent=2,
        sort_keys=True,
    )
)

print("\nMANIFEST STATUS")
print(
    "Phase:",
    evaluation_manifest.get("phase"),
)
print(
    "Test inference passes:",
    evaluation_manifest.get(
        "test_inference_passes"
    ),
)
print(
    "Test SHA-256:",
    evaluation_manifest.get("test_sha256"),
)
print(
    "Artifact count:",
    len(
        evaluation_manifest.get(
            "artifact_hashes",
            {},
        )
    ),
)

assert state["status"] == "completed"
assert state["test_inference_passes"] == 1
assert (
    evaluation_manifest["phase"]
    == "locked_test_evaluation"
)
assert (
    evaluation_manifest["test_inference_passes"]
    == 1
)
assert (
    saved_metrics["test_inference_passes"]
    == 1
)
assert (
    state["test_sha256"]
    == evaluation_manifest["test_sha256"]
)
assert state["test_sha256"] == (
    "edf650024fc2f74c5f3eea1bc04c3b9"
    "09c52884849067987196fd8b795bb43ff"
)

print(
    "\nEVALUASI TELAH SELESAI TEPAT SATU KALI."
)
print("JANGAN JALANKAN RUNNER KEMBALI.")

Evaluation directory: /content/drive/MyDrive/SIPATURE/evaluation/20260801_indobert-silver-v1_locked-test-v1
State tersedia: True
Metrics tersedia: True
Manifest tersedia: True

EVALUATION STATE
{
  "calibration_sha256": "f7efc6525c2b7de9351b31257beb7455db7f3b1c7abe675a412038505123eadb",
  "completed_at": "2026-08-01T11:51:53.591558+00:00",
  "split_manifest_sha256": "2044659130e0f2a80bc6c15bd75052ecce51c8a064f8c24ef8eb86d06110a2c0",
  "started_at": "2026-08-01T11:51:46.585146+00:00",
  "status": "completed",
  "test_access_policy": "Do not rerun after failure; document a methodological incident.",
  "test_inference_passes": 1,
  "test_sha256": "edf650024fc2f74c5f3eea1bc04c3b909c52884849067987196fd8b795bb43ff"
}

MANIFEST STATUS
Phase: locked_test_evaluation
Test inference passes: 1
Test SHA-256: edf650024fc2f74c5f3eea1bc04c3b909c52884849067987196fd8b795bb43ff
Artifact count: 11

EVALUASI TELAH SELESAI TEPAT SATU KALI.
JANGAN JALANKAN RUNNER KEMBALI.


In [30]:
import json

metrics_path = EVALUATION_DIR / "metrics.json"

assert metrics_path.is_file()

saved_metrics = json.loads(
    metrics_path.read_text(encoding="utf-8")
)

aspect = saved_metrics["aspect"]
polarity = saved_metrics["polarity"]
probability_quality = saved_metrics["calibration"]
inference_result = saved_metrics["inference"]

print("Metrics berhasil dimuat dari:", metrics_path)
print("Phase:", saved_metrics["phase"])
print(
    "Test inference passes:",
    saved_metrics["test_inference_passes"],
)
print(
    "Inference fields:",
    sorted(inference_result),
)

assert (
    saved_metrics["phase"]
    == "locked_test_evaluation"
)
assert saved_metrics["test_inference_passes"] == 1

print("\nMetrik dimuat tanpa inferensi ulang.")

Metrics berhasil dimuat dari: /content/drive/MyDrive/SIPATURE/evaluation/20260801_indobert-silver-v1_locked-test-v1/metrics.json
Phase: locked_test_evaluation
Test inference passes: 1
Inference fields: ['aspect_latency_seconds', 'device', 'model_loading_excluded_from_inference_latency', 'model_loading_seconds', 'polarity_latency_seconds']

Metrik dimuat tanpa inferensi ulang.


In [32]:
print("LOCKED-TEST METRICS TERSIMPAN")
print("==============================")

print("\nASPECT")
print("Macro F1:", aspect["macro_f1"])
print("Micro F1:", aspect["micro_f1"])
print(
    "Total aspect latency:",
    inference_result.get("aspect_latency_seconds"),
    "detik",
)
print(
    "Aspect latency per record:",
    aspect["latency"]["milliseconds_per_record"],
    "ms",
)

print("\nPER-LABEL ASPECT")
for label, result in aspect["per_label"].items():
    print(
        f"- {label}: "
        f"F1={result['f1']:.6f}, "
        f"precision={result['precision']:.6f}, "
        f"recall={result['recall']:.6f}, "
        f"support={result['support']}, "
        f"predicted_support={result['predicted_support']}"
    )

print("\nPRECISION AT ALERT - OVERALL MICRO")
print(
    json.dumps(
        aspect["precision_at_alert"]["overall_micro"],
        indent=2,
        sort_keys=True,
    )
)

print("\nPRECISION AT ALERT - PER LABEL")
for label, result in aspect[
    "precision_at_alert"
]["per_label"].items():
    print(
        f"- {label}: "
        f"threshold={result['threshold']}, "
        f"precision={result['precision']}, "
        f"predicted_support={result['predicted_support']}, "
        f"validation_target_met="
        f"{result['validation_target_met']}"
    )

print("\nPROBABILITY QUALITY")
print("Method:", probability_quality["method"])
print("ECE:", probability_quality["ece"])
print("Brier:", probability_quality["brier"])

print("\nPOLARITY")
print("Macro F1:", polarity["macro_f1"])
print("Support:", polarity["support"])
print("Labels:", polarity["labels"])
print("Confusion matrix:")

for row in polarity["confusion_matrix"]:
    print(row)

print(
    "Polarity latency:",
    inference_result.get("polarity_latency_seconds"),
    "detik",
)
print(
    "Model loading:",
    inference_result.get("model_loading_seconds"),
    "detik",
)
print(
    "Device:",
    inference_result.get("device"),
)
print(
    "Model loading excluded:",
    inference_result.get(
        "model_loading_excluded_from_inference_latency"
    ),
)

print("\nSEVERITY")
print(
    json.dumps(
        saved_metrics["severity"],
        indent=2,
        sort_keys=True,
    )
)

print("\nBASELINE COMPARISON")
for model in saved_metrics[
    "baseline_comparison"
]["models"]:
    print(
        f"- {model['model']}: "
        f"Macro F1={model['macro_f1']}, "
        f"Micro F1={model['micro_f1']}"
    )

print("\nMetrik dibaca dari artifact tersimpan.")
print("Tidak ada inferensi ulang.")

import hashlib

manifest_path = EVALUATION_DIR / "manifest.json"

evaluation_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

def evaluation_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

artifact_hashes = evaluation_manifest[
    "artifact_hashes"
]

print(
    "Jumlah artifact dalam manifest:",
    len(artifact_hashes),
)

verified_count = 0

for relative_path, expected_hash in sorted(
    artifact_hashes.items()
):
    artifact_path = EVALUATION_DIR / relative_path

    assert artifact_path.is_file(), (
        f"Artifact evaluasi hilang: {relative_path}"
    )

    actual_hash = evaluation_sha256(artifact_path)
    matches = actual_hash == expected_hash

    print(
        f"- {relative_path}:",
        "COCOK" if matches else "TIDAK COCOK",
    )
    print("  SHA-256:", actual_hash)

    assert matches, (
        f"Hash artifact tidak cocok: {relative_path}"
    )

    verified_count += 1

assert verified_count == 11

print(
    "\nSeluruh 11 artifact evaluasi cocok "
    "dengan manifest."
)
print("Tidak ada inferensi ulang.")

LOCKED-TEST METRICS TERSIMPAN

ASPECT
Macro F1: 0.52472918092202
Micro F1: 0.5241379310344828
Total aspect latency: 1.7108013119998304 detik
Aspect latency per record: 8.469313425741735 ms

PER-LABEL ASPECT
- access: F1=0.530612, precision=0.419355, recall=0.722222, support=18, predicted_support=31
- cleanliness: F1=0.566667, precision=0.500000, recall=0.653846, support=26, predicted_support=34
- comfort: F1=0.358209, precision=0.255319, recall=0.600000, support=20, predicted_support=47
- crowding: F1=0.571429, precision=0.500000, recall=0.666667, support=6, predicted_support=8
- maintenance: F1=0.551724, precision=0.421053, recall=0.800000, support=10, predicted_support=19
- opening_hours: F1=0.000000, precision=0.000000, recall=0.000000, support=2, predicted_support=12
- parking: F1=0.833333, precision=1.000000, recall=0.714286, support=21, predicted_support=15
- price_transparency: F1=0.563380, precision=0.512821, recall=0.625000, support=32, predicted_support=39
- public_facilities

In [33]:
evaluation_manifest_sha256 = evaluation_sha256(
    EVALUATION_DIR / "manifest.json"
)
evaluation_metrics_sha256 = evaluation_sha256(
    EVALUATION_DIR / "metrics.json"
)
evaluation_state_sha256 = evaluation_sha256(
    EVALUATION_DIR / "evaluation-state.json"
)

print(
    "Evaluation manifest SHA-256:",
    evaluation_manifest_sha256,
)
print(
    "Evaluation metrics SHA-256:",
    evaluation_metrics_sha256,
)
print(
    "Evaluation state SHA-256:",
    evaluation_state_sha256,
)

assert evaluation_manifest["test_sha256"] == (
    "edf650024fc2f74c5f3eea1bc04c3b9"
    "09c52884849067987196fd8b795bb43ff"
)
assert (
    evaluation_manifest["test_inference_passes"]
    == 1
)

print("\nHash final evaluasi telah dicatat.")
print("Evaluasi tidak dijalankan ulang.")

Evaluation manifest SHA-256: d58fd1c17af3c0e0c5de2b118fc70072b5a8190b2bf999082f35d3003975fc88
Evaluation metrics SHA-256: 923a000e43c9f6528ac53a5c3b99827cfd0ed55ec38db5f3c9a2564f3db0f9da
Evaluation state SHA-256: 1b237b33f32b044fac0607c1c412292a5ea278555b05000417773377dd44c81f

Hash final evaluasi telah dicatat.
Evaluasi tidak dijalankan ulang.


In [34]:
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

EVIDENCE_ID = (
    "20260801_indobert-silver-v1_a8-evidence"
)

EVIDENCE_DIR = (
    Path("/content/drive/MyDrive/SIPATURE/evidence")
    / EVIDENCE_ID
)

EVIDENCE_ZIP_BASE = (
    Path("/content/drive/MyDrive/SIPATURE/evidence")
    / EVIDENCE_ID
)

assert not EVIDENCE_DIR.exists(), (
    "Evidence directory sudah ada. "
    "Jangan menimpa evidence yang telah dibuat."
)
assert not EVIDENCE_ZIP_BASE.with_suffix(".zip").exists()

EVIDENCE_DIR.mkdir(parents=True)

safe_sources = {
    "calibration/calibration.json": (
        CALIBRATION_DIR / "calibration.json"
    ),
    "calibration/manifest.json": (
        CALIBRATION_DIR / "manifest.json"
    ),
    "calibration/calibration.png": (
        CALIBRATION_DIR / "calibration.png"
    ),
    "calibration/freeze-receipt.json": (
        FREEZE_RECEIPT
    ),
    "evaluation/metrics.json": (
        EVALUATION_DIR / "metrics.json"
    ),
    "evaluation/manifest.json": (
        EVALUATION_DIR / "manifest.json"
    ),
    "evaluation/evaluation-state.json": (
        EVALUATION_DIR / "evaluation-state.json"
    ),
    "evaluation/aspect-per-label-f1.png": (
        EVALUATION_DIR / "aspect-per-label-f1.png"
    ),
    "evaluation/comparison.png": (
        EVALUATION_DIR / "comparison.png"
    ),
    "evaluation/polarity-confusion-matrix.png": (
        EVALUATION_DIR
        / "polarity-confusion-matrix.png"
    ),
    "evaluation/test-probability-quality.png": (
        EVALUATION_DIR
        / "test-probability-quality.png"
    ),
}

for relative_path, source in safe_sources.items():
    assert source.is_file(), f"File tidak ditemukan: {source}"

    destination = EVIDENCE_DIR / relative_path
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    shutil.copy2(source, destination)

print("Safe evidence files disalin:", len(safe_sources))

Safe evidence files disalin: 11


In [35]:
a8_summary = {
    "evidence_id": EVIDENCE_ID,
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked_test_completed_once",
    "test_inference_passes": 1,
    "test_sha256": evaluation_manifest[
        "test_sha256"
    ],
    "reference_label_type": (
        "ai_assisted_weak_supervision_silver"
    ),
    "model_run_id": (
        "20260801-1024_indobert-silver-v1"
    ),
    "calibration_id": CALIBRATION_ID,
    "evaluation_id": EVALUATION_ID,
    "calibration": {
        "temperature": 0.6,
        "validation_macro_f1_threshold_0_50": (
            temporary_validation_metrics["macro_f1"]
        ),
        "validation_macro_f1_tuned": (
            tuned_validation_metrics["macro_f1"]
        ),
        "validation_micro_f1_tuned": (
            tuned_validation_metrics["micro_f1"]
        ),
        "nll_before": frozen_calibration[
            "validation_calibration"
        ]["nll_before"],
        "nll_after": frozen_calibration[
            "validation_calibration"
        ]["nll_after"],
        "ece_before": frozen_calibration[
            "validation_calibration"
        ]["ece_before"],
        "ece_after": frozen_calibration[
            "validation_calibration"
        ]["ece_after"],
        "brier_before": frozen_calibration[
            "validation_calibration"
        ]["brier_before"],
        "brier_after": frozen_calibration[
            "validation_calibration"
        ]["brier_after"],
    },
    "locked_test": {
        "aspect_macro_f1": aspect["macro_f1"],
        "aspect_micro_f1": aspect["micro_f1"],
        "precision_at_alert_overall_micro": (
            aspect["precision_at_alert"][
                "overall_micro"
            ]
        ),
        "ece": probability_quality["ece"],
        "brier": probability_quality["brier"],
        "polarity_macro_f1": polarity["macro_f1"],
        "polarity_support": polarity["support"],
        "severity_status": saved_metrics[
            "severity"
        ]["status"],
        "aspect_latency_ms_per_record": (
            aspect["latency"][
                "milliseconds_per_record"
            ]
        ),
    },
    "baseline_comparison": saved_metrics[
        "baseline_comparison"
    ],
    "artifact_hashes": {
        "calibration_json": calibration_sha256,
        "calibration_manifest": (
            calibration_manifest_sha256
        ),
        "freeze_receipt": freeze_receipt_sha256,
        "evaluation_manifest": (
            evaluation_manifest_sha256
        ),
        "evaluation_metrics": (
            evaluation_metrics_sha256
        ),
        "evaluation_state": (
            evaluation_state_sha256
        ),
    },
    "excluded_restricted_files": [
        "test_silver_v1.jsonl",
        "validation-predictions.npz",
        "test-predictions.npz",
        "errors.json",
        "errors.csv",
        "audit-fp.json",
        "audit-fn.json",
    ],
    "limitations": [
        (
            "Metrics measure silver agreement, "
            "not human-gold performance."
        ),
        (
            "Keyword rules and silver labels share "
            "taxonomy vocabulary and may have "
            "correlated errors."
        ),
        (
            "Severity is unavailable because A7 did "
            "not produce a supported severity model."
        ),
        (
            "The locked test result must not be used "
            "to retune model configuration or thresholds."
        ),
    ],
}

summary_path = EVIDENCE_DIR / "summary.json"

summary_path.write_text(
    json.dumps(
        a8_summary,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("Summary:", summary_path)

Summary: /content/drive/MyDrive/SIPATURE/evidence/20260801_indobert-silver-v1_a8-evidence/summary.json


In [36]:
def evidence_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

inventory = {
    "evidence_id": EVIDENCE_ID,
    "files": {},
}

for path in sorted(EVIDENCE_DIR.rglob("*")):
    if not path.is_file():
        continue

    relative_path = str(
        path.relative_to(EVIDENCE_DIR)
    )

    inventory["files"][relative_path] = {
        "bytes": path.stat().st_size,
        "sha256": evidence_sha256(path),
    }

inventory_path = EVIDENCE_DIR / "inventory.json"

inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

# Inventory ditambahkan terakhir, lalu hash-nya dicatat
# secara terpisah agar tidak membuat self-reference.
inventory_sha256 = evidence_sha256(inventory_path)

zip_path = Path(
    shutil.make_archive(
        str(EVIDENCE_ZIP_BASE),
        "zip",
        root_dir=EVIDENCE_DIR,
    )
)

zip_sha256 = evidence_sha256(zip_path)

print("Evidence directory:", EVIDENCE_DIR)
print("Jumlah safe evidence files:", len(
    [
        path
        for path in EVIDENCE_DIR.rglob("*")
        if path.is_file()
    ]
))
print("Inventory SHA-256:", inventory_sha256)
print("Evidence ZIP:", zip_path)
print("Evidence ZIP size:", f"{zip_path.stat().st_size:,}", "byte")
print("Evidence ZIP SHA-256:", zip_sha256)

assert zip_path.is_file()

print("\nA8 SAFE EVIDENCE BUNDLE BERHASIL DIBUAT.")
print("Restricted review-level files tidak disertakan.")

Evidence directory: /content/drive/MyDrive/SIPATURE/evidence/20260801_indobert-silver-v1_a8-evidence
Jumlah safe evidence files: 13
Inventory SHA-256: 3f20451fb93bc214715247e92f97a08007f6fdaf8afef703fac451fc56a2fe06
Evidence ZIP: /content/drive/MyDrive/SIPATURE/evidence/20260801_indobert-silver-v1_a8-evidence.zip
Evidence ZIP size: 213,689 byte
Evidence ZIP SHA-256: 069731af3e8e52761accc5416b50e3d7055aabdef2880eeebacc471f4fbc326b

A8 SAFE EVIDENCE BUNDLE BERHASIL DIBUAT.
Restricted review-level files tidak disertakan.


In [37]:
import zipfile

with zipfile.ZipFile(zip_path) as archive:
    zip_names = sorted(archive.namelist())

print("Isi ZIP:")

for name in zip_names:
    print("-", name)

for forbidden_name in [
    "test_silver_v1.jsonl",
    "validation-predictions.npz",
    "test-predictions.npz",
    "errors.json",
    "errors.csv",
    "audit-fp.json",
    "audit-fn.json",
]:
    assert not any(
        name.endswith(forbidden_name)
        for name in zip_names
    ), forbidden_name

print("\nZIP tidak memuat file review-level restricted.")

Isi ZIP:
- calibration/
- calibration/calibration.json
- calibration/calibration.png
- calibration/freeze-receipt.json
- calibration/manifest.json
- evaluation/
- evaluation/aspect-per-label-f1.png
- evaluation/comparison.png
- evaluation/evaluation-state.json
- evaluation/manifest.json
- evaluation/metrics.json
- evaluation/polarity-confusion-matrix.png
- evaluation/test-probability-quality.png
- inventory.json
- summary.json

ZIP tidak memuat file review-level restricted.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!test -d /content/hackathon/.git || git clone REPLACE_REPOSITORY_URL /content/hackathon
!git -C /content/hackathon pull --ff-only
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .

## Phase 1: validation only
Set controlled Drive source paths below. Copy only the manifest and validation file for Phase 1; the locked test remains in Drive. The model run must be the full A7 Drive run.

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

from sipature_ml.evaluation import run_calibration

SPLIT_DIR = Path('/content/a8-splits')
DRIVE_SPLIT_DIR = Path('/content/drive/MyDrive/SIPATURE/frozen-splits')
MANIFEST_SOURCE = DRIVE_SPLIT_DIR / 'split_manifest_silver_v1.json'
VALIDATION_SOURCE = DRIVE_SPLIT_DIR / 'validation_silver_v1.jsonl'
LOCKED_TEST_SOURCE = DRIVE_SPLIT_DIR / 'test_silver_v1.jsonl'
MODEL_RUN_DIR = Path('/content/drive/MyDrive/SIPATURE/runs/REPLACE_A7_RUN_ID')
CALIBRATION_DIR = Path('/content/drive/MyDrive/SIPATURE/calibration/REPLACE_CALIBRATION_ID')
SPLIT_DIR.mkdir(exist_ok=False)
shutil.copy2(MANIFEST_SOURCE, SPLIT_DIR / MANIFEST_SOURCE.name)
shutil.copy2(VALIDATION_SOURCE, SPLIT_DIR / VALIDATION_SOURCE.name)
manifest = json.loads((SPLIT_DIR / MANIFEST_SOURCE.name).read_text())
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
assert sha256(SPLIT_DIR / VALIDATION_SOURCE.name) == manifest['outputs']['validation']['sha256']
print({'manifest_sha256': sha256(SPLIT_DIR / MANIFEST_SOURCE.name), 'validation_sha256': sha256(SPLIT_DIR / VALIDATION_SOURCE.name)})
calibration = run_calibration(SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR)
assert calibration['test_read'] is False
calibration

## Mandatory pause
Stop here. Inspect `calibration.json` and `manifest.json`, record their hashes, and freeze the directory. Do not continue until a human confirms the artifact is accepted. Only after that confirmation may the locked test file be copied from controlled Drive storage to `SPLIT_DIR`; do not use browser upload or inspect its contents.

In [ ]:
CONFIRMATION_PHRASE = 'I AUTHORIZE ONE LOCKED TEST ACCESS'
confirmation = input(f'Type exactly: {CONFIRMATION_PHRASE}\n')
assert confirmation == CONFIRMATION_PHRASE, 'Locked test access is not authorized'

## Phase 2: one locked-test pass
After the pause, copy the hash-locked test from controlled Drive into `SPLIT_DIR`, then execute this cell once. Use one predeclared output ID. Never choose a fresh ID after any attempted locked-test access; preserve `evaluation-state.json` and document an incident.

In [ ]:
from sipature_ml.evaluation import run_locked_test_evaluation

assert confirmation == CONFIRMATION_PHRASE
shutil.copy2(LOCKED_TEST_SOURCE, SPLIT_DIR / LOCKED_TEST_SOURCE.name)
EVALUATION_DIR = Path('/content/drive/MyDrive/SIPATURE/evaluation/REPLACE_EVALUATION_ID')
BASELINE_METRICS_DIR = Path('/content/hackathon/ml/artifacts/metrics')
metrics = run_locked_test_evaluation(
    SPLIT_DIR, MODEL_RUN_DIR, CALIBRATION_DIR, EVALUATION_DIR, BASELINE_METRICS_DIR
)
assert metrics['test_inference_passes'] == 1
metrics